In [1]:
# ─── Cell 1: Import + 학습 설정 (CFG) ────────────────────────────────────────
import os
import math
import random
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from modules import *

# ── Tokenizer 경고 제거 ────────────────────────────────────────────────────────
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── 캐시 경로 고정 (/data3/llm_download) ─────────────────────────────────────
CACHE_ROOT = "/data3/llm_download"
os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = f"{CACHE_ROOT}/datasets"

# ── 하이퍼파라미터 ─────────────────────────────────────────────────────────────
CFG = dict(
    # ── 모델 (참고 레포 기본값 기준) ────────────────────────────────────────────
    vocab_size      = 50280,   # config_mamba 기본값
    d_model         = 768,     # config_mamba 기본값
    n_layer         = 24,      # config_mamba 기본값
    d_state         = 128,     # SpikeMamba2 기본값
    expand          = 2,       # SpikeMamba2 기본값
    headdim         = 64,      # SpikeMamba2 기본값
    conv1d_kernel   = 4,
    use_gated_mlp   = False,
    sgc_indices     = None,    # None → [0, n_layer//2, n_layer-1]
    spike_qmin      = -4.0,
    spike_qmax      = 4.0,
    # ── 멀티 GPU 설정 ─────────────────────────────────────────────────────────
    gpu_ids         = [3,4,5],
    # ── 데이터 ────────────────────────────────────────────────────────────────
    block_size      = 256,     # distill_smamba.yaml max_seq_length
    batch_size      = 3,       # distill_smamba.yaml per_device_train_batch_size
    # ── 학습 (참고 레포 distill_smamba.yaml 기준) ─────────────────────────────
    n_epochs        = 5,
    lr              = 1e-3,
    warmup_ratio    = 0.01,
    weight_decay    = 0.0,
    grad_clip       = 1.0,
    # ── Loss 가중치 ───────────────────────────────────────────────────────────
    ce_weight       = 1.0,     # distill_smamba.yaml
    sgc_weight      = 1.0,     # reference의 kl_weight에 대응시킨 값
    # ── 기타 ──────────────────────────────────────────────────────────────────
    device          = "auto",
    seed            = 42,
    log_interval    = 10,
    save_dir        = "./ckpt_smamba",
)

if CFG["device"] == "auto":
    if torch.cuda.is_available() and torch.cuda.device_count() > max(CFG["gpu_ids"]):
        CFG["device"] = f"cuda:{CFG['gpu_ids'][0]}"
    else:
        CFG["device"] = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
print(f"Device : {CFG['device']}")
if torch.cuda.is_available():
    print(f"Visible CUDA count={torch.cuda.device_count()}")
    print(f"Requested GPU IDs={CFG['gpu_ids']}")
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE={os.environ['HF_DATASETS_CACHE']}")
print(f"d_model={CFG['d_model']}  n_layer={CFG['n_layer']}  "
      f"d_inner={CFG['d_model']*CFG['expand']}  "
      f"nheads={CFG['d_model']*CFG['expand']//CFG['headdim']}")


Device : cuda:3
Visible CUDA count=6
Requested GPU IDs=[3, 4, 5]
HF_HOME=/data3/llm_download
HF_DATASETS_CACHE=/data3/llm_download/datasets
d_model=768  n_layer=24  d_inner=1536  nheads=24


/home/bhkim003/anaconda3/envs/spbllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ─── Cell 2: Tokenizer + Dataset + DataLoader ────────────────────────────────
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/gpt-neox-20b",
    cache_dir=os.environ["HF_HOME"],
)
tokenizer.pad_token = tokenizer.eos_token
CFG["vocab_size"] = len(tokenizer)   # 50280
print(f"Tokenizer: EleutherAI/gpt-neox-20b  vocab={len(tokenizer)}")

# ── Dataset: wikitext-2 ───────────────────────────────────────────────────────
raw = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    cache_dir=os.environ["HF_DATASETS_CACHE"],
)

def tokenize_and_chunk(split, block_size):
    """전체 텍스트를 이어 붙인 뒤 (block_size+1) 크기 청크로 분할"""
    text = "\n\n".join(t for t in raw[split]["text"] if t.strip())
    ids  = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(ids) - block_size, block_size):
        chunks.append(ids[i : i + block_size + 1])
    return chunks

class LMDataset(Dataset):
    def __init__(self, chunks):
        self.data = [torch.tensor(c, dtype=torch.long) for c in chunks]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        seq = self.data[idx]
        return seq[:-1], seq[1:]   # input, label  (각각 block_size 길이)

block_size = CFG["block_size"]
train_chunks = tokenize_and_chunk("train",      block_size)
val_chunks   = tokenize_and_chunk("validation", block_size)

train_ds = LMDataset(train_chunks)
val_ds   = LMDataset(val_chunks)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, pin_memory=True, num_workers=2)

print(f"Train: {len(train_ds):,} chunks  |  Val: {len(val_ds):,} chunks")
print(f"Train steps / epoch: {len(train_loader)}")


Tokenizer: EleutherAI/gpt-neox-20b  vocab=50277
Train: 9,499 chunks  |  Val: 986 chunks
Train steps / epoch: 3167


In [3]:
# ─── Cell 3: 모델 인스턴스화 (+ optional DataParallel) ───────────────────────
base_model = SpikingMamba(
    vocab_size    = CFG["vocab_size"],
    d_model       = CFG["d_model"],
    n_layer       = CFG["n_layer"],
    d_state       = CFG["d_state"],
    expand        = CFG["expand"],
    conv1d_kernel = CFG["conv1d_kernel"],
    headdim       = CFG["headdim"],
    mlp_ratio     = 2,
    use_gated_mlp = CFG["use_gated_mlp"],
    spike_qmin    = CFG["spike_qmin"],
    spike_qmax    = CFG["spike_qmax"],
    proj_bias     = False,   # 레포와 동일
    sgc_indices   = CFG["sgc_indices"],
)

use_dp = (
    torch.cuda.is_available()
    and CFG["device"].startswith("cuda:")
    and len(CFG["gpu_ids"]) > 1
    and torch.cuda.device_count() > max(CFG["gpu_ids"]) 
)
if use_dp:
    primary = CFG["gpu_ids"][0]
    base_model = base_model.to(f"cuda:{primary}")
    model = torch.nn.DataParallel(
        base_model,
        device_ids=CFG["gpu_ids"],
        output_device=primary,
    )
else:
    model = base_model.to(CFG["device"])

model_core = model.module if isinstance(model, torch.nn.DataParallel) else model

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 파라미터: {total_p:,}  (학습 가능: {train_p:,})")

sgc_idxs = [i for i, blk in enumerate(model_core.blocks) if blk.sgc_on]
print(f"SGC on 블록: {sgc_idxs}")
print(f"DataParallel={isinstance(model, torch.nn.DataParallel)}")
if isinstance(model, torch.nn.DataParallel):
    print(f"GPU IDs={CFG['gpu_ids']}  primary=cuda:{CFG['gpu_ids'][0]}")
print(f"d_inner={CFG['d_model']*CFG['expand']}  nheads={CFG['d_model']*CFG['expand']//CFG['headdim']}")


총 파라미터: 128,981,184  (학습 가능: 128,981,184)
SGC on 블록: [0, 12, 23]
DataParallel=True
GPU IDs=[3, 4, 5]  primary=cuda:3
d_inner=1536  nheads=24


In [4]:
# ─── Cell 5: SGC Loss + Optimizer + Scheduler ───────────────────────────────

def sgc_alignment_loss(sgc_outputs):
    """
    smooth-spike mutual alignment loss.
    레포 mamba_kd_trainer.py 공식:
      L = ||softmax(smooth).detach - softmax(spike)||_2
        + ||softmax(smooth) - softmax(spike).detach||_2
    → in_proj 출력과 out_proj 출력 각각 적용 후 SGC 블록 수로 나눔.

    sgc_outputs: list of (proj_spike, proj_smooth, y_spike, y_smooth)
      proj_*: (B, L, 2E+2N+H)  — in_proj 출력
      y_*:    (B, L, D)         — out_proj 출력 (residual 합산 전)
    """
    total = 0.0
    for proj_s, proj_f, y_s, y_f in sgc_outputs:
        # in_proj 정렬
        sp_f = F.softmax(proj_f, dim=-1)
        sp_s = F.softmax(proj_s, dim=-1)
        total += torch.norm(sp_f.detach() - sp_s, p=2, dim=-1).mean()
        total += torch.norm(sp_f - sp_s.detach(), p=2, dim=-1).mean()
        # out_proj 정렬
        so_f = F.softmax(y_f, dim=-1)
        so_s = F.softmax(y_s, dim=-1)
        total += torch.norm(so_f.detach() - so_s, p=2, dim=-1).mean()
        total += torch.norm(so_f - so_s.detach(), p=2, dim=-1).mean()
    # 레포: hidden_proj_loss / 3  (SGC 블록 기본 3개 기준)
    return total / max(len(sgc_outputs), 1)


# 임베딩 포함 전체 파라미터 학습
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"\n학습 가능 파라미터: {sum(p.numel() for p in trainable):,}")

optimizer = torch.optim.AdamW(
    trainable,
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    betas=(0.9, 0.95),
)

total_steps  = len(train_loader) * CFG["n_epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

from transformers import get_cosine_schedule_with_warmup
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"총 스텝: {total_steps}  |  warmup: {warmup_steps}  |  lr_peak: {CFG['lr']:.1e}")



학습 가능 파라미터: 128,981,184
총 스텝: 15835  |  warmup: 158  |  lr_peak: 1.0e-03


In [ ]:
# ─── Cell 5: Training Loop ───────────────────────────────────────────────────
import time
os.makedirs(CFG["save_dir"], exist_ok=True)

def safe_ppl(loss_value, max_exp=80.0):
    """exp overflow/포화 방지용 PPL 계산"""
    if loss_value >= max_exp:
        return float("inf")
    return math.exp(loss_value)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    eval_start_time = time.time()
    total_eval_steps = len(loader)
    eval_log_interval = max(1, total_eval_steps // 10)  # 10번 정도 진행상황 출력

    print(f"  [Eval] started: total_steps={total_eval_steps}")
    for eval_step, (x, y) in enumerate(loader, start=1):
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

        if eval_step % eval_log_interval == 0 or eval_step == total_eval_steps:
            elapsed = time.time() - eval_start_time
            print(f"  [Eval] step {eval_step}/{total_eval_steps} | elapsed={elapsed//60:.0f}m{elapsed%60:.0f}s")

    avg_loss = total_loss / total_n
    return avg_loss, safe_ppl(avg_loss)

device    = CFG["device"]
best_val  = float("inf")
step      = 0
total_train_steps = len(train_loader) * CFG["n_epochs"]
epoch_start_time = None

for epoch in range(CFG["n_epochs"]):
    epoch_start_time = time.time()
    model.train()
    run_ce = run_sgc = run_tot = 0.0

    print(f"\n{'='*70}")
    print(f"Epoch {epoch+1}/{CFG['n_epochs']} started...")
    print(f"{'='*70}")

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)   # (B, L)

        logits, sgc_out = model(x)           # logits: (B,L,vocab), sgc_out: list

        # 첫 스텝에서 로그잇 스케일 점검
        if epoch == 0 and batch_idx == 0:
            print(
                f"[Debug] logits mean={logits.mean().item():.4f} std={logits.std().item():.4f} "
                f"min={logits.min().item():.4f} max={logits.max().item():.4f}"
            )

        # CE loss
        loss_ce = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
        )

        # SGC loss (smooth-spike 정렬, 레포 mamba_kd_trainer 공식)
        if sgc_out:
            loss_sgc = sgc_alignment_loss(sgc_out)
        else:
            loss_sgc = torch.tensor(0.0, device=device)

        loss = CFG["ce_weight"] * loss_ce + CFG["sgc_weight"] * loss_sgc

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            (p for p in model.parameters() if p.requires_grad),
            CFG["grad_clip"],
        )
        optimizer.step()
        scheduler.step()

        run_ce  += loss_ce.item()
        run_sgc += loss_sgc.item()
        run_tot += loss.item()
        step    += 1

        if (batch_idx + 1) % CFG["log_interval"] == 0:
            n = CFG["log_interval"]
            elapsed = time.time() - epoch_start_time
            avg_ce = run_ce / n
            avg_ppl = safe_ppl(avg_ce)
            avg_bpc = avg_ce / math.log(2)
            lr_now = scheduler.get_last_lr()[0]
            ppl_str = f"{avg_ppl:.3e}" if math.isfinite(avg_ppl) else "inf(>exp(80))"

            print(f"[Epoch {epoch+1}] Step {batch_idx+1}/{len(train_loader)} | "
                  f"CE={avg_ce:.4f} | SGC={run_sgc/n:.4f} | PPL={ppl_str} | BPC={avg_bpc:.2f} | "
                  f"LR={lr_now:.2e} | Time={elapsed//60:.0f}m{elapsed%60:.0f}s")
            run_ce = run_sgc = run_tot = 0.0

    # ── Validation ───────────────────────────────────────────────────────────
    epoch_elapsed = time.time() - epoch_start_time
    print(f"\n[Epoch {epoch+1}] Training done ({epoch_elapsed//60:.0f}m{epoch_elapsed%60:.0f}s)")
    print(f"  Running validation...")

    val_loss, val_ppl = evaluate(model, val_loader, device)
    val_ppl_str = f"{val_ppl:.3e}" if math.isfinite(val_ppl) else "inf(>exp(80))"
    print(f"  Val Loss={val_loss:.4f}  Val PPL={val_ppl_str}")

    if val_loss < best_val:
        best_val  = val_loss
        ckpt_path = os.path.join(CFG["save_dir"], "best.pt")
        model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
        torch.save({"epoch": epoch + 1, "model": model_to_save.state_dict(),
                    "cfg": CFG, "val_loss": val_loss}, ckpt_path)
        print(f"  ✓ Checkpoint saved: {ckpt_path}")

print(f"\n{'='*70}")
best_ppl = safe_ppl(best_val)
best_ppl_str = f"{best_ppl:.3e}" if math.isfinite(best_ppl) else "inf(>exp(80))"
print(f"학습 완료!  best val_loss={best_val:.4f}  ppl={best_ppl_str}")
print(f"{'='*70}")


Epoch 1/5 started...
[Debug] logits mean=0.0033 std=27.7334 min=-145.3239 max=357.7709
[Epoch 1] Step 10/3167 | CE=258.3504 | SGC=0.1537 | PPL=inf(>exp(80)) | BPC=372.72 | LR=6.33e-05 | Time=2m3s
[Epoch 1] Step 20/3167 | CE=159.6993 | SGC=0.1599 | PPL=inf(>exp(80)) | BPC=230.40 | LR=1.27e-04 | Time=4m2s
[Epoch 1] Step 30/3167 | CE=71.0397 | SGC=0.1972 | PPL=7.114e+30 | BPC=102.49 | LR=1.90e-04 | Time=5m55s
[Epoch 1] Step 40/3167 | CE=57.5004 | SGC=0.1970 | PPL=9.378e+24 | BPC=82.96 | LR=2.53e-04 | Time=7m54s
[Epoch 1] Step 50/3167 | CE=52.6277 | SGC=0.1843 | PPL=7.177e+22 | BPC=75.93 | LR=3.16e-04 | Time=9m47s
[Epoch 1] Step 60/3167 | CE=50.6087 | SGC=0.1898 | PPL=9.530e+21 | BPC=73.01 | LR=3.80e-04 | Time=11m40s
[Epoch 1] Step 70/3167 | CE=47.0557 | SGC=0.1827 | PPL=2.729e+20 | BPC=67.89 | LR=4.43e-04 | Time=13m38s
[Epoch 1] Step 80/3167 | CE=47.8370 | SGC=0.1752 | PPL=5.961e+20 | BPC=69.01 | LR=5.06e-04 | Time=15m37s
[Epoch 1] Step 90/3167 | CE=46.0909 | SGC=0.1773 | PPL=1.040e+20 |

In [ ]:
# ─── Cell 7: Greedy / Top-k Generation ───────────────────────────────────────
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60, temperature=1.0, top_k=50):
    """Greedy(top_k=1) 또는 top-k 샘플링으로 텍스트 생성"""
    device = CFG["device"]
    model.eval()
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new_tokens):
        logits, _ = model(ids)
        logits = logits[:, -1, :] / temperature        # (1, vocab)
        if top_k > 1:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
        else:
            next_id = logits.argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0].tolist(), skip_special_tokens=True)

# ── 실행 ───────────────────────────────────────────────────────────────────────
val_loss, val_ppl = evaluate(model, val_loader, CFG["device"])
print(f"최종  val_loss={val_loss:.4f}  PPL={val_ppl:.2f}")

for prompt in [
    "The future of artificial intelligence is",
    "Once upon a time in a land far away",
]:
    print(f"\n[Prompt]    {prompt}")
    print(f"[Generated] {generate(model, prompt)}")


  [Eval] started: total_steps=986
  [Eval] step 98/986 | elapsed=0m34s
  [Eval] step 196/986 | elapsed=1m8s
  [Eval] step 294/986 | elapsed=1m42s
  [Eval] step 392/986 | elapsed=2m17s
  [Eval] step 490/986 | elapsed=2m52s
  [Eval] step 588/986 | elapsed=3m26s
  [Eval] step 686/986 | elapsed=4m1s
  [Eval] step 784/986 | elapsed=4m42s
  [Eval] step 882/986 | elapsed=5m16s
  [Eval] step 980/986 | elapsed=5m52s
  [Eval] step 986/986 | elapsed=5m54s
최종  val_loss=6.6135  PPL=745.12

[Prompt]    The future of artificial intelligence is
[Generated] The future of artificial intelligence is still : the It is located . The album debuted at number four on the Billboard 200 by the Royal Navy in the final , including the Pacific Ocean previous hit was one of the National Institute of the first round of August , he received the team between top and her sister a week and early 20th centuries

[Prompt]    Once upon a time in a land far away
[Generated] Once upon a time in a land far away to be complete

In [ ]:
# ─── Debug Cell: One-batch loss/logit scale check ─────────────────────────────
with torch.no_grad():
    model.eval()
    x_dbg, y_dbg = next(iter(train_loader))
    x_dbg, y_dbg = x_dbg.to(CFG["device"]), y_dbg.to(CFG["device"])
    logits_dbg, _ = model(x_dbg)
    ce_dbg = F.cross_entropy(logits_dbg.reshape(-1, logits_dbg.size(-1)), y_dbg.reshape(-1))
    print(f"[Debug] one-batch CE={ce_dbg.item():.4f}")
    print(
        f"[Debug] logits mean={logits_dbg.mean().item():.4f} std={logits_dbg.std().item():.4f} "
        f"min={logits_dbg.min().item():.4f} max={logits_dbg.max().item():.4f}"
    )
    print(f"[Debug] current LR={scheduler.get_last_lr()[0]:.2e}")

[Debug] one-batch CE=6.1592
[Debug] logits mean=-50.9969 std=5.9417 min=-67.3603 max=66.2846
[Debug] current LR=5.63e-07


In [ ]:
import sys
print("python:", sys.executable)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

import transformers
print("transformers:", transformers.__version__)

# 핵심: transformers가 torch를 보고 있는지
from transformers.utils import is_torch_available
print("is_torch_available:", is_torch_available())

python: /home/bhkim003/anaconda3/envs/spbllm/bin/python
torch: 2.1.1 | cuda: True


/home/bhkim003/anaconda3/envs/spbllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 4.51.3
is_torch_available: True
